# Mortality reconstruction across weather lag windows

Compare three 1–30-day sweeps: the hybrid, weather-only with air temperature,
and weather-only with air temperature and dew point. Day 9 is the manuscript
reference. The six- and nine-day hybrid runs use 50 members each; other runs
use their ten available members.

Reconstruct mortality once per member from October 2014 through December 2025.
The MSE figure averages **member-level MSE** over the saved training and
validation splits. The period tables average **predictions first**, then
calculate RMSE between those ensemble means and registered deaths. Weekly
predictions sum Monday–Sunday and use registered national weekly totals.
Calendar-year bins assign weeks by their Thursday, and the seasonal definitions
match notebook 04. Run the full sweep on HPC.

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys

cwd = Path.cwd()
repo = next(
    (
        path for path in (cwd, *cwd.parents)
        if (path / "configs" / "training.example.yaml").is_file()
    ),
    None,
)
if repo is None:
    raise FileNotFoundError("Open the notebook from the sari-weather-mortality checkout.")

sys.path.insert(0, str(repo))
sys.path.insert(0, str(repo / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load helpers from this checkout even if Jupyter cached another notebooks package.
utils_path = repo / "notebooks" / "utils.py"
spec = spec_from_file_location("sari_notebook_utils", utils_path)
notebook_utils = module_from_spec(spec)
spec.loader.exec_module(notebook_utils)

ANALYSIS_PERIOD = notebook_utils.ANALYSIS_PERIOD
daily_index = notebook_utils.daily_index
run_checkpoint_paths = notebook_utils.run_checkpoint_paths
ensemble = notebook_utils.ensemble
national_daily = notebook_utils.national_daily
weekly_sum = notebook_utils.weekly_sum
DEVICE = notebook_utils.DEVICE

from sari_t.inference import InferenceModel
from sari_t.run.checkpoint import load_checkpoint, load_run_config, restore_predictor

from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle

## Load models for inspection

The next cell loads member 000 for the selected window from each setup onto CPU.
Inspect `model` for the hybrid predictor or `models[setup]` for another predictor.

In [ ]:
inspection_member = 0
inspection_window = 9
setups = {
    "Hybrid (temperature + dew point)": "temperature_kernel_hybrid",
    "Weather only (temperature)": "temperature_kernel_weather_only",
    "Weather only (temperature + dew point)": "temperature_kernel_weather_only_tas_tdew",
}
windows = range(1, 31)
sweeps = {}
for label, group in setups.items():
    runs = {}
    for run in sorted((repo / "runtime" / "training" / "sets" / group).iterdir()):
        run = run.resolve()
        config = load_run_config(run / "config.yaml")
        lag = config.model_config.kernel_T_days
        if lag in runs:
            raise ValueError(f"Multiple {lag}-day runs in {group}")
        paths = run_checkpoint_paths(run)
        runs[lag] = (config, paths)
    if set(runs) != set(windows):
        raise ValueError(f"{group} must contain one run for each window in {list(windows)}")
    sweeps[label] = runs

models = {
    label: restore_predictor(
        load_checkpoint(runs[inspection_window][1][inspection_member]), device="cpu",
    ).eval()
    for label, runs in sweeps.items()
}
model = models["Hybrid (temperature + dew point)"]
model

## Reconstruct once and calculate MSE and RMSE

In [ ]:
dates = daily_index()


def period_masks(dates):
    years = dates.year.to_numpy()
    iso_weeks = dates.isocalendar().week.to_numpy(dtype=int)
    return {
        "Overall": np.ones(len(dates), dtype=bool),
        "Influenza\nseason": (iso_weeks >= 40) | (iso_weeks <= 20),
        "Summer": (iso_weeks >= 15) & (iso_weeks <= 38),
        "Pre-pandemic\n2014–2019": years <= 2019,
        "Pandemic\n2020–2022": (years >= 2020) & (years <= 2022),
        "2023–2024": (years >= 2023) & (years <= 2024),
        "2025": years == 2025,
        "2023–2025": years >= 2023,
    }

# The Thursday determines the ISO week and its calendar-period assignment.
if len(dates) % 7 or dates[0].dayofweek != 0:
    raise ValueError("Expected complete Monday–Sunday analysis weeks.")
week_dates = dates[3::7]
daily_masks = period_masks(dates)
weekly_masks = period_masks(week_dates)


def period_rmse(predicted, observed, masks):
    squared_errors = (predicted - observed).square().numpy()
    return {window: np.sqrt(squared_errors[mask].mean())
            for window, mask in masks.items()}


def national_counts(workspace, result):
    counts = national_daily(workspace, result)
    if result.labels.weekly_start_date != dates[0].date():
        raise ValueError("The registered weekly series must start on the first analysis Monday.")
    counts["Observed weekly"] = result.labels.weekly_de_age_sex.sum(dim=(1, 2))
    return counts





mse_rows = []
daily_rmse_tables = {}
weekly_rmse_tables = {}
expected_split = None
observed = observed_weekly = None
for setup, runs in sweeps.items():
    daily_rows = []
    weekly_rows = []
    for window in windows:
        config, paths = runs[window]
        training_start = config.period.start_date
        validation_end = config.period.end_date
        weeks = ((validation_end - training_start).days + 1) // 7
        validation_weeks = int(weeks * config.training_config.validation_fraction)
        if not 0 < validation_weeks < weeks:
            raise ValueError(f"{setup}, {window} days has no usable training/validation split.")
        split_position = (training_start - ANALYSIS_PERIOD.start_date).days
        validation_position = split_position + 7 * (weeks - validation_weeks)
        end_position = (validation_end - ANALYSIS_PERIOD.start_date).days + 1
        split_key = (split_position, validation_position, end_position)
        if expected_split is None:
            expected_split = split_key
        elif split_key != expected_split:
            raise ValueError("All lag runs must use the same training and validation dates.")
        if not 0 <= split_position < validation_position < end_position <= len(dates) or any(
            position % 7 for position in split_key
        ):
            raise ValueError("Training and validation must be complete weeks in the analysis period.")

        print(f"{setup}, {window} input days, {len(paths)} members", flush=True)
        inference_model = InferenceModel.from_checkpoint(
            paths[0], data_config=config.data_config, period=ANALYSIS_PERIOD, device=DEVICE,
        )
        members = ensemble(inference_model, paths, national_counts, keep_members=True)
        del inference_model

        registered = members["Observed"][0]
        registered_weekly = members["Observed weekly"][0]
        if registered.numel() != len(dates) or registered_weekly.numel() != len(week_dates):
            raise ValueError(f"Unexpected mortality window for {setup}, {window} days")
        if observed is None:
            observed, observed_weekly = registered, registered_weekly
        elif not registered.equal(observed) or not registered_weekly.equal(observed_weekly):
            raise ValueError(f"Different observed deaths for {setup}, {window} days")

        daily_predictions = members["Predicted"]
        weekly_predictions = daily_predictions.reshape(len(paths), -1, 7).sum(dim=2)
        daily_squared_error = (daily_predictions - observed).square()
        weekly_squared_error = (weekly_predictions - observed_weekly).square()
        for split, start, end in (
            ("Training", split_position, validation_position),
            ("Validation", validation_position, end_position),
        ):
            for frequency, errors, positions in (
                ("Daily", daily_squared_error, slice(start, end)),
                ("Weekly", weekly_squared_error, slice(start // 7, end // 7)),
            ):
                losses = errors[:, positions].mean(dim=1)
                mse_rows.extend((setup, window, int(path.stem.split("_")[-1]), split, frequency, loss)
                                for path, loss in zip(paths, losses.tolist()))

        daily_rows.append(period_rmse(daily_predictions.mean(dim=0), observed, daily_masks))
        weekly_rows.append(period_rmse(weekly_predictions.mean(dim=0), observed_weekly, weekly_masks))

    lag_index = pd.Index(windows, name="Weather input days")
    daily_rmse_tables[setup] = pd.DataFrame(daily_rows, index=lag_index)
    weekly_rmse_tables[setup] = pd.DataFrame(weekly_rows, index=lag_index)

mse_by_member = pd.DataFrame(mse_rows, columns=["Setup", "Window (days)", "Member",
                                                "Split", "Frequency", "MSE"])
mse_by_window = mse_by_member.groupby(["Setup", "Window (days)", "Split", "Frequency"])[
    "MSE"
].agg(["mean", "min", "max"]).reset_index()

## Daily and weekly mortality MSE across lag windows

Rows show model setups. Daily and weekly MSE are on the left and right, respectively. Models share the y-scale within each column. Breaks omit large gaps between the plotted ranges. Lines show the mean across members and shading shows the member minimum and maximum.

In [ ]:
def mse_axis_bands(frame):
    """Split the shared scale at the largest gap between curve envelopes."""
    intervals = sorted(
        (group["min"].min(), group["max"].max())
        for _, group in frame.groupby(["Setup", "Split"])
    )
    low = intervals[0][0]
    high = max(end for _, end in intervals)
    span = high - low
    padding = 0.04 * span if span else max(1, abs(high) * 0.04)
    limits = (max(0, low - padding), high + padding)
    gaps = []
    covered_end = intervals[0][1]
    for start, end in intervals[1:]:
        if start > covered_end:
            gaps.append((covered_end, start))
        covered_end = max(covered_end, end)
    if not gaps:
        return [limits]
    lower_end, upper_start = max(gaps, key=lambda gap: gap[1] - gap[0])
    if upper_start - lower_end < 0.22 * span:
        return [limits]
    return [(upper_start - padding, limits[1]), (limits[0], lower_end + padding)]


split_styles = {
    "Training": ("#2A5C9E", "-"),
    "Validation": ("#D73027", "--"),
}
frequencies = ("Daily", "Weekly")
panels = {frequency: mse_by_window.loc[mse_by_window["Frequency"] == frequency]
          for frequency in frequencies}
bands_by_frequency = {frequency: mse_axis_bands(panel) for frequency, panel in panels.items()}

with plt.rc_context({"font.family": "DejaVu Serif", "axes.axisbelow": True}):
    fig = plt.figure(figsize=(14, 3.5 * len(setups)))
    grid = fig.add_gridspec(len(setups), 2, left=0.12, right=0.98,
                           bottom=0.08, top=0.87, hspace=0.44, wspace=0.32)
    shared_y = {}
    shared_x = None
    for row, setup in enumerate(setups):
        for col, frequency in enumerate(frequencies):
            bands = bands_by_frequency[frequency]
            panel = grid[row, col].subgridspec(len(bands), 1, hspace=0.12)
            selected = panels[frequency].loc[panels[frequency]["Setup"] == setup]
            for band, limits in enumerate(bands):
                ax = fig.add_subplot(panel[band, 0], sharex=shared_x,
                                     sharey=shared_y.get((col, band)))
                shared_y.setdefault((col, band), ax)
                if shared_x is None:
                    shared_x = ax

                for split, (color, linestyle) in split_styles.items():
                    group = selected.loc[selected["Split"] == split].sort_values("Window (days)")
                    x = group["Window (days)"].to_numpy()
                    ax.plot(x, group["mean"], color=color, linestyle=linestyle,
                            linewidth=1.5, label=split)
                    ax.fill_between(x, group["min"], group["max"], color=color,
                                    alpha=0.2, linewidth=0)
                ax.axvline(9, color="0.5", linestyle=":", linewidth=1,
                           label="Nine-day reference")
                ax.set_ylim(*limits)
                ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
                last_band = band == len(bands) - 1
                ax.tick_params(axis="x", labelbottom=row == len(setups) - 1 and last_band)
                if band == 0:
                    ax.set_title(f"{frequency} MSE\n{setup}" if row == 0 else setup, fontsize=11)
                if row == len(setups) - 1 and last_band:
                    ax.set_xlabel("Weather input window (days)")
                if len(bands) == 2:
                    edge = 0 if band == 0 else 1
                    ax.spines["bottom" if band == 0 else "top"].set_visible(False)
                    if band == 0:
                        ax.tick_params(axis="x", bottom=False)
                    ax.plot([0, 1], [edge, edge], transform=ax.transAxes,
                            marker=[(-1, -0.5), (1, 0.5)], markersize=8,
                            linestyle="none", color="0.25", clip_on=False)

    handles, labels = shared_x.get_legend_handles_labels()
    fig.suptitle("Mortality reconstruction error across weather lag windows", y=0.98, fontsize=16)
    fig.supylabel("MSE (deaths²)", x=0.018)
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.945),
               ncol=3, frameon=False)
    plt.show()

## Daily RMSE by period and model

In [ ]:
def plot_rmse_tables(tables, resolution, unit):
    values = np.stack([table.to_numpy() for table in tables.values()])
    if not np.isfinite(values).all():
        raise ValueError("The RMSE tables contain non-finite values.")

    norm = Normalize(vmin=values.min(), vmax=values.max())
    cmap = plt.get_cmap("magma_r")
    reference_window = 9
    with plt.rc_context({"font.family": "DejaVu Serif"}):
        for setup, table in tables.items():
            data = table.to_numpy()
            fig, ax = plt.subplots(figsize=(12, 12), layout="constrained")
            heatmap = ax.imshow(data, cmap=cmap, norm=norm, aspect="auto")
            ax.set_title(f"{setup} · {resolution} RMSE by period")
            ax.set_ylabel(table.index.name)
            ax.set_xticks(np.arange(len(table.columns)), table.columns)
            ax.set_yticks(np.arange(len(table.index)), table.index)
            ax.xaxis.tick_top()
            ax.tick_params(axis="both", length=0, labelsize=9)
            ax.tick_params(axis="x", pad=7)

            for (row, col), value in np.ndenumerate(data):
                red, green, blue, _ = cmap(norm(value))
                luminance = 0.2126 * red + 0.7152 * green + 0.0722 * blue
                ax.text(col, row, f"{value:.1f}", ha="center", va="center",
                        color="black" if luminance > 0.5 else "white", fontsize=8)

            reference_row = table.index.get_loc(reference_window)
            ax.add_patch(Rectangle((-0.49, reference_row - 0.49), len(table.columns) - 0.02,
                                   0.98, fill=False, edgecolor="#2A5C9E", linewidth=1.6))
            ax.get_yticklabels()[reference_row].set_fontweight("bold")
            fig.colorbar(heatmap, ax=ax, shrink=0.8, label=f"RMSE (deaths/{unit})")
            plt.show()


plot_rmse_tables(daily_rmse_tables, "Daily", "day")

## Weekly RMSE by period and model

In [ ]:
plot_rmse_tables(weekly_rmse_tables, "Weekly", "week")